In [1]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px

# Get datasets

In [2]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)
df_normal['nature'] = 'non-equivalent'

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)
df_equiv['nature'] = 'equivalent'


In [3]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [4]:
def print_box_plot(df, cat, file_name, type):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['nature'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['nature'] = df['nature'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(#scatter
        df, 
        y=type, 
        x="x_offset", 
        color="nature", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0, 
        boxgap=0
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, "Visualisation of the mutant detectability score for each metric", f'results/TEST_NEW/', file_name, yaxis_range=[0, 1])


In [5]:
threshold = "N"
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 


df_hw = df[df['hardware'] == 'kyiv']
selected_columns = df_hw[["metric", 'nature', 'ideal_distance']]
file_name = f'visu_ideal'
print_box_plot(selected_columns, "metric", file_name, 'ideal_distance')

for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    # df_metric = df_hw[df_hw['metric'] == m]
    # df_threshold = df_hw[df_hw['threshold'] == threshold]
    
    #for key, categories in c.table_data.items():
    selected_columns = df_hw[["metric", 'nature', 'noisy_distance']]
    file_name = f'visu_{hw}'
    print_box_plot(selected_columns, "metric", file_name, 'noisy_distance')
        

In [9]:
for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    equivs = df_hw.loc[df_hw['nature'] == 'equivalent']
    
    print(f"=================== {hw} ===================")
    r = df_hw.shape[0]
    print(f"Rows: {r}\n")

    for m in ['T','E','H','J']: #c.metrics:
        tmp = equivs.loc[equivs['metric'] == m]
        tot = tmp.shape[0]
        tmp_ideal = tmp.loc[tmp['ideal_distance'] >= 0.9]
        outliers = tmp_ideal.shape[0]
        if outliers > 0:
            print(tmp_ideal)
        
        # Save DataFrame to CSV
        csv_path = f'results/TEST_NEW/ideal_outliers_{hw}.csv'
        tmp_ideal.to_csv(csv_path, mode='w', header=True, index=False)

=================== kyiv ===================
Rows: 3886880

         gates  depth  singlequbit_gates  multiqubit_gates          Input  \
15879       36     16                 13                23     Quratest_3   
15882       36     16                 13                23    PureState_5   
15886       36     16                 13                23    PureState_7   
15888       36     16                 13                23    PureState_8   
15896       36     16                 13                23   PureState_12   
...        ...    ...                ...               ...            ...   
1103970     47     23                 15                32  PureState_121   
1103971     47     23                 15                32   Quratest_121   
1103974     47     23                 15                32  PureState_123   
1103976     47     23                 15                32  PureState_124   
1103978     47     23                 15                32  PureState_125   

        Input_t

In [10]:
for hw in c.hardware:
    df_hw = df[df['hardware'] == hw]
    selected_columns = df_hw[["metric", 'nature', 'ideal_distance', 'noisy_distance']]
    equivs = selected_columns.loc[selected_columns['nature'] == 'equivalent']
    normals = selected_columns.loc[selected_columns['nature'] == 'non-equivalent']
    
    print(f"=================== {hw} ===================")
    r = df_hw.shape[0]
    print(f"Rows: {r}\n")

    for m in ['T']: #c.metrics:
        tmp = equivs.loc[equivs['metric'] == m]
        tot = tmp.shape[0]
        tmp_ideal = tmp.loc[tmp['ideal_distance'] >= 0.9]
        outliers = tmp_ideal.shape[0]
        print(tmp_ideal)
        print(f"IDEAL: Metric {m} as {outliers} equiv. mutants above 0.9 out of {tot} mutants")
        
        tmp_noisy = tmp.loc[tmp['noisy_distance'] >= 0.9]
        outliers = tmp_noisy.shape[0]
        print(f"NOISY: Metric {m} as {outliers} equiv. mutants above 0.9 out of {tot} mutants")
        
        
        tmp = normals.loc[normals['metric'] == m]
        tot = tmp.shape[0]
        tmp_ideal = tmp.loc[tmp['ideal_distance'] >= 0.9]
        outliers = tmp_ideal.shape[0]
        print(f"IDEAL: Metric {m} as {outliers} non-equiv. mutants above 0.9 out of {tot} mutants")
    
        tmp_noisy = tmp.loc[tmp['noisy_distance'] >= 0.9]
        outliers = tmp_noisy.shape[0]
        print(f"NOISY: Metric {m} as {outliers} non-equiv. mutants above 0.9 out of {tot} mutants")
        

=================== kyiv ===================
Rows: 3886880

Empty DataFrame
Columns: [metric, nature, ideal_distance, noisy_distance]
Index: []
IDEAL: Metric T as 0 equiv. mutants above 0.9 out of 448864 mutants
NOISY: Metric T as 0 equiv. mutants above 0.9 out of 448864 mutants
IDEAL: Metric T as 11412 non-equiv. mutants above 0.9 out of 328512 mutants
NOISY: Metric T as 10740 non-equiv. mutants above 0.9 out of 328512 mutants
=================== brisbane ===================
Rows: 3886880

Empty DataFrame
Columns: [metric, nature, ideal_distance, noisy_distance]
Index: []
IDEAL: Metric T as 0 equiv. mutants above 0.9 out of 448864 mutants
NOISY: Metric T as 0 equiv. mutants above 0.9 out of 448864 mutants
IDEAL: Metric T as 11412 non-equiv. mutants above 0.9 out of 328512 mutants
NOISY: Metric T as 10860 non-equiv. mutants above 0.9 out of 328512 mutants
=================== sherbrooke ===================
Rows: 3881760

Empty DataFrame
Columns: [metric, nature, ideal_distance, noisy_di